#1. Load and clean metadata from cvs file.

In [30]:

import pandas as pd
import numpy as np
import wfdb
import ast
import os

#(using own path)
path = ''

#loading one row metadata per ECG
metadata= pd.read_csv(path + 'ptbxl_database.csv', index_col='ecg_id')
metadata.scp_codes = metadata.scp_codes.apply(lambda x: ast.literal_eval(x))

#removing rows with more than 60% missing values, (60% er bare noe jeg tippet vi kunne bruke, sikkert alt fra 55% til 70%)
threshold = 0.6
metadata = metadata[metadata.columns[metadata.isnull().mean() < threshold]]
metadata = metadata.loc[metadata.isnull().mean(axis=1) < threshold]


#2. Creating diagnostic labels (y)
(fra 40 til 5 hovedklasser)

In [31]:
#scp codes to the 5 diagnostic classes: 
#NORM (normal), CD (conduction disturbance), HYP (hypertrophy), MI (myocardial infarction), STTC (ST-T changes)
agg_df = pd.read_csv(path + 'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

metadata['diagnostic_superclass'] = metadata.scp_codes.apply(diagnostic)

#removing rows with no diagnostic superclass
metadata = metadata[metadata.diagnostic_superclass.apply(len) > 0]

y = metadata.diagnostic_superclass
print(len(y))

21388



#3. Loading ECG signals (X)

In [ ]:

def signals(df):
    data = [wfdb.rdsamp(path + f)[0]for f in df.filename_lr]
    return np.array(data)
#loading signal files. (tok meg 10 min: så da slipper vi gjøre det hver gang)
os.makedirs('data/processed', exist_ok=True)
if os.path.exists('data/processed/X_100hz.npy'):
    X = np.load('data/processed/X_100hz.npy')
else:
    X = signals(metadata)
    np.save('data/processed/X_100hz.npy', X)

print(X.shape, len(y))


(21388, 1000, 12) 21388


#4. Explore(ikke veldig viktig, bare gir oversikt)


In [33]:
print(y.head(10))

print("\nECG per class:")
print(y.explode().value_counts())

multi = y[y.apply(len) > 1]
print("\n", len(multi), "ECGs with multiple diagnostic superclasses")

ecg_id
1     [NORM]
2     [NORM]
3     [NORM]
4     [NORM]
5     [NORM]
6     [NORM]
7     [NORM]
8       [MI]
9     [NORM]
10    [NORM]
Name: diagnostic_superclass, dtype: object

ECG per class:
diagnostic_superclass
NORM    9514
MI      5469
STTC    5235
CD      4898
HYP     2649
Name: count, dtype: int64

 5144 ECGs with multiple diagnostic superclasses


In [34]:
multi = y[y.apply(len) > 1]
print(len(multi), "ECG with more than 1 diagnostic superclass")

print(multi.head())

5144 ECG with more than 1 diagnostic superclass
ecg_id
39    [MI, STTC]
45     [HYP, CD]
50      [MI, CD]
77      [MI, CD]
87    [STTC, CD]
Name: diagnostic_superclass, dtype: object
